# SOP Model Evaluation: Base Llama vs. LoRA Adapter

This notebook evaluates `meta-llama/Llama-3.2-1B-Instruct` and the SOP LoRA adapter on the same 100-example validation split. It reports **ROUGE-L** and **BERTScore F1**, saves every prediction, and displays qualitative comparisons.

> Run this notebook on a Kaggle or Colab GPU. You need access to the gated Llama model, `val.jsonl` from `01_data_preparation.ipynb`, and the `sop_lora_adapter` produced by `02_model_training.ipynb`.

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.45.0 accelerate==0.34.0 peft==0.13.0 datasets==2.21.0 rouge-score bert-score pandas tqdm

In [ ]:
import gc
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from bert_score import score as bert_score
from datasets import load_dataset
from peft import PeftModel
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configure inputs

Update the two paths if your Kaggle dataset or Colab upload uses different locations. `MAX_EXAMPLES=None` evaluates all 100 validation examples; use a small number for a quick smoke test.

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
VAL_PATH = Path("/kaggle/input/sop-training-data/val.jsonl")
ADAPTER_PATH = Path("/kaggle/input/sop-lora-adapter/sop_lora_adapter")
OUTPUT_DIR = Path("/kaggle/working/sop_evaluation")
MAX_EXAMPLES = None  # Set to 5 for a quick smoke test

GENERATION_CONFIG = {
    "max_new_tokens": 850,
    "temperature": 0.7,
    "top_p": 0.9,
    "do_sample": True,
}

assert VAL_PATH.exists(), f"Validation file not found: {VAL_PATH}"
assert ADAPTER_PATH.exists(), f"LoRA adapter not found: {ADAPTER_PATH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Validation data: {VAL_PATH}")
print(f"Adapter: {ADAPTER_PATH}")

If the model is gated, authenticate with a Hugging Face token that has accepted the Llama 3.2 license.

In [ ]:
from huggingface_hub import login

login()

## 3. Load the validation set

In [ ]:
validation = load_dataset("json", data_files=str(VAL_PATH), split="train")
if MAX_EXAMPLES is not None:
    validation = validation.select(range(min(MAX_EXAMPLES, len(validation))))

def split_example(example):
    messages = example["messages"]
    prompt_messages = [message for message in messages if message["role"] != "assistant"]
    references = [message["content"] for message in messages if message["role"] == "assistant"]
    if len(references) != 1:
        raise ValueError(f"Expected one assistant reference, found {len(references)}")
    return prompt_messages, references[0]

prepared_examples = [split_example(example) for example in validation]
references = [reference for _, reference in prepared_examples]
print(f"Evaluation examples: {len(prepared_examples)}")
print(f"Reference characters (median): {int(np.median([len(text) for text in references]))}")

## 4. Generate predictions

The base model is evaluated first and then deleted before the LoRA model is loaded. This keeps the notebook within a T4's memory. Both runs use the same prompts, generation settings, and per-example random seeds.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def load_model(adapter_path=None):
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    if adapter_path is not None:
        model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()
    return model

def generate_predictions(model, examples, run_seed=SEED):
    predictions = []
    for index, (prompt_messages, _) in enumerate(tqdm(examples)):
        set_seed(run_seed + index)
        prompt = tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                pad_token_id=tokenizer.eos_token_id,
                **GENERATION_CONFIG,
            )
        generated_tokens = output[0, inputs["input_ids"].shape[1]:]
        predictions.append(tokenizer.decode(generated_tokens, skip_special_tokens=True).strip())
    return predictions

def release_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
print("Generating base-model predictions...")
base_model = load_model()
base_predictions = generate_predictions(base_model, prepared_examples)
release_model(base_model)

print("Generating fine-tuned predictions...")
tuned_model = load_model(ADAPTER_PATH)
tuned_predictions = generate_predictions(tuned_model, prepared_examples)
release_model(tuned_model)

assert len(base_predictions) == len(tuned_predictions) == len(references)
print("Generation complete.")

## 5. Compute ROUGE-L and BERTScore

In [ ]:
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l_scores(predictions, targets):
    return [
        rouge.score(target, prediction)["rougeL"].fmeasure
        for prediction, target in zip(predictions, targets)
    ]

base_rouge = rouge_l_scores(base_predictions, references)
tuned_rouge = rouge_l_scores(tuned_predictions, references)

_, _, base_bert_f1 = bert_score(
    base_predictions, references, lang="en", verbose=True, rescale_with_baseline=True
)
_, _, tuned_bert_f1 = bert_score(
    tuned_predictions, references, lang="en", verbose=True, rescale_with_baseline=True
)

base_bert = base_bert_f1.cpu().numpy()
tuned_bert = tuned_bert_f1.cpu().numpy()

summary = pd.DataFrame(
    [
        {"Model": "Base Llama 3.2 1B", "ROUGE-L": np.mean(base_rouge), "BERTScore F1": np.mean(base_bert)},
        {"Model": "+ LoRA (ours)", "ROUGE-L": np.mean(tuned_rouge), "BERTScore F1": np.mean(tuned_bert)},
    ]
)
summary.style.format({"ROUGE-L": "{:.4f}", "BERTScore F1": "{:.4f}"})

## 6. Save reproducible results

In [ ]:
records = pd.DataFrame(
    {
        "example_id": range(len(references)),
        "reference": references,
        "base_prediction": base_predictions,
        "tuned_prediction": tuned_predictions,
        "base_rouge_l": base_rouge,
        "tuned_rouge_l": tuned_rouge,
        "base_bertscore_f1": base_bert,
        "tuned_bertscore_f1": tuned_bert,
    }
)

records.to_json(OUTPUT_DIR / "predictions.jsonl", orient="records", lines=True, force_ascii=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
with open(OUTPUT_DIR / "run_config.json", "w") as file:
    json.dump(
        {
            "base_model": BASE_MODEL,
            "adapter_path": str(ADAPTER_PATH),
            "validation_path": str(VAL_PATH),
            "examples": len(references),
            "seed": SEED,
            "generation_config": GENERATION_CONFIG,
        },
        file,
        indent=2,
    )

print(f"Saved results to {OUTPUT_DIR}")
display(summary)

## 7. Inspect side-by-side samples

Automatic overlap metrics do not fully measure writing quality. Review these examples for structure, specificity, coherence, repetition, and unresolved placeholders.

In [ ]:
sample_ids = np.linspace(0, len(records) - 1, num=min(5, len(records)), dtype=int)
for sample_id in sample_ids:
    row = records.iloc[sample_id]
    print("=" * 100)
    print(f"EXAMPLE {sample_id}")
    print("\nREFERENCE\n", row["reference"][:1200])
    print("\nBASE MODEL\n", row["base_prediction"][:1200])
    print("\nFINE-TUNED MODEL\n", row["tuned_prediction"][:1200])
    print(
        f"\nScores — base ROUGE-L: {row['base_rouge_l']:.4f}, tuned ROUGE-L: {row['tuned_rouge_l']:.4f}"
    )

## Interpretation checklist

Before reporting the final comparison:

- confirm all 100 validation examples were evaluated;
- keep generation settings identical between models;
- report the seed and library versions;
- include the aggregate table and 3–5 qualitative examples;
- discuss metric limitations: SOPs can be valid even when wording differs from the reference;
- check whether fine-tuning reduces or increases template placeholders.